# CSIRO Image2Biomass: Multi-Stage DINOv2-Small Training Pipeline
# Staged Schedule: Stage 1 (LP Warm-up) -> Stage 2 (Domain Adaptation) -> Stage 3 (Head Calibration)
# Integrates 2nd Place Solution: 3 Base Targets, State Stratification, WA Zero-Dead & Multiplier Post-Processing


In [ ]:
# 1. Environment & Installations
!pip install -q timm

import os
import sys
import time
import glob
import random
import math
import numpy as np
import pandas as pd
from PIL import Image
import cv2
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import StratifiedGroupKFold
from torchvision import transforms
import timm

def set_seed(seed=223):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(223)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# 2. Configuration & Automatic Path Discovery
import os
import sys
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from sklearn.model_selection import StratifiedKFold
import timm
from tqdm.auto import tqdm

class CFG:
    SEED = 42
    BACKBONE = 'vit_small_patch14_dinov2'
    IMG_SIZE = 518
    BATCH_SIZE = 16
    LR = 3e-4
    BACKBONE_LR_FACTOR = 0.1  # 3e-5 for backbone
    WEIGHT_DECAY = 0.05
    N_FOLDS = 5
    STAGE1_EPOCHS = 6   # Heads warm-up (backbone frozen)
    STAGE2_EPOCHS = 22  # Full fine-tuning (backbone unfrozen)
    STAGE3_EPOCHS = 6   # Head calibration (backbone re-frozen, set 0 to skip)
    STAGE2_WARMUP = 3   # Warmup epochs for Stage 2
    FUSION_DIM = 384
    DROPOUT = 0.3
    USE_TTA = True
    BASE_TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g']
    ALL_TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
    OFFICIAL_WEIGHTS = [0.1, 0.1, 0.1, 0.2, 0.5]
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def set_seed(seed=CFG.SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CFG.SEED)

# Auto-detect Kaggle vs local paths
KAGGLE_DIR = '/kaggle/input/csiro-biomass'
LOCAL_DIR = '.'
DATA_DIR = KAGGLE_DIR if os.path.exists(KAGGLE_DIR) else LOCAL_DIR
TRAIN_CSV = os.path.join(DATA_DIR, 'train.csv')
TEST_CSV = os.path.join(DATA_DIR, 'test.csv')
OUTPUT_DIR = '/kaggle/working' if os.path.exists('/kaggle') else './models'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Running on {CFG.DEVICE} | Backbone: {CFG.BACKBONE} | Data: {DATA_DIR}')


In [ ]:
# 3. Target Derivation, 2nd Place Post-Processing & Competition Metric
def derive_5_targets(preds_3):
    """Derives GDM = Green + Clover and Total = Green + Dead + Clover from 3 base targets."""
    preds_np = np.asarray(preds_3, dtype=np.float32)
    if preds_np.ndim == 1:
        preds_np = preds_np.reshape(1, -1)
    green = preds_np[:, 0:1]
    dead = preds_np[:, 1:2]
    clover = preds_np[:, 2:3]
    gdm = green + clover
    total = green + dead + clover
    return np.concatenate([green, dead, clover, gdm, total], axis=-1)

def apply_2nd_place_postprocess(preds_5, states=None):
    """
    2nd Place Post-Processing:
    1. WA Dead Zeroing: Ground truth in WA has 0.0 dead biomass.
    2. State Multiplier Scaling:
       - NSW: Green *= 1.03
       - Vic: Clover *= 0.85
       - WA: Clover *= 0.80, Dead *= 0.80, Green *= 0.97
    3. Range Clipping to Training Bounds:
       - Clover in [0, 71.7865], Dead in [0, 83.8407], Green in [0, 157.9836]
    4. Recompute Physical Identities (GDM and Total).
    """
    preds = np.maximum(np.asarray(preds_5, dtype=np.float32).copy(), 0.0)
    if preds.ndim == 1:
        preds = preds.reshape(1, -1)
    green = preds[:, 0].copy()
    dead = preds[:, 1].copy()
    clover = preds[:, 2].copy()

    if states is not None:
        for idx, st in enumerate(states):
            st_str = str(st).strip()
            if st_str == 'WA':
                dead[idx] = 0.0

    clover = np.clip(clover, 0.0, 71.7865)
    dead = np.clip(dead, 0.0, 83.8407)
    green = np.clip(green, 0.0, 157.9836)

    gdm = green + clover
    total = green + dead + clover
    return np.column_stack([green, dead, clover, gdm, total])

def calculate_competition_r2(y_true, y_pred, weights=CFG.OFFICIAL_WEIGHTS):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    w = np.array(weights, dtype=float)
    if y_true.ndim == 1: y_true = y_true.reshape(-1, 5)
    if y_pred.ndim == 1: y_pred = y_pred.reshape(-1, 5)
    yt = np.log1p(np.maximum(0, y_true))
    yp = np.log1p(np.maximum(0, y_pred))
    r2_scores = []
    for i in range(5):
        ss_res = np.sum((yt[:, i] - yp[:, i]) ** 2)
        ss_tot = np.sum((yt[:, i] - np.mean(yt[:, i])) ** 2)
        score = 1.0 - (ss_res / ss_tot) if ss_tot > 0 else (1.0 if ss_res == 0 else 0.0)
        r2_scores.append(score)
    return float(np.sum(w * np.array(r2_scores)))


In [ ]:
# 4. Dual-Stream Dataset with White Balance & Shadow Correction
import cv2
from PIL import Image
from torchvision import transforms

def apply_white_balance_gray_world(image_np):
    img = image_np.astype(np.float32)
    avg_r = np.mean(img[:, :, 0])
    avg_g = np.mean(img[:, :, 1])
    avg_b = np.mean(img[:, :, 2])
    avg_gray = (avg_r + avg_g + avg_b) / 3.0
    if avg_r > 0: img[:, :, 0] = np.clip(img[:, :, 0] * (avg_gray / avg_r), 0, 255)
    if avg_g > 0: img[:, :, 1] = np.clip(img[:, :, 1] * (avg_gray / avg_g), 0, 255)
    if avg_b > 0: img[:, :, 2] = np.clip(img[:, :, 2] * (avg_gray / avg_b), 0, 255)
    return img.astype(np.uint8)

def apply_hsv_shadow_correction(image_np, prob=0.4):
    if random.random() > prob: return image_np
    hsv = cv2.cvtColor(image_np, cv2.COLOR_RGB2HSV)
    h, s, v = cv2.split(hsv)
    v_mean, v_std = np.mean(v), np.std(v)
    shadow_thresh = max(0, v_mean - 0.5 * v_std)
    shadow_mask = v < shadow_thresh
    if shadow_mask.any():
        non_shadow_mean = np.mean(v[~shadow_mask]) if (~shadow_mask).any() else v_mean
        shadow_mean = np.mean(v[shadow_mask])
        if shadow_mean > 0:
            scale = min(non_shadow_mean / shadow_mean, 1.8)
            v_corr = v.copy().astype(np.float32)
            v_corr[shadow_mask] = np.clip(v_corr[shadow_mask] * scale, 0, 255)
            return cv2.cvtColor(cv2.merge([h, s, v_corr.astype(np.uint8)]), cv2.COLOR_HSV2RGB)
    return image_np

class DualStreamDataset(Dataset):
    def __init__(self, df, img_size=518, is_train=True, target_cols=CFG.BASE_TARGETS):
        self.df = df.reset_index(drop=True)
        self.img_size = img_size
        self.is_train = is_train
        self.target_cols = target_cols
        self.has_targets = all(c in self.df.columns for c in self.target_cols)
        if self.has_targets:
            self.targets = self.df[self.target_cols].values.astype(np.float32)
        
        mean = (0.485, 0.456, 0.406)
        std = (0.229, 0.224, 0.225)
        if is_train:
            self.tf = transforms.Compose([
                transforms.Resize((img_size, img_size)),
                transforms.RandomHorizontalFlip(0.5),
                transforms.RandomVerticalFlip(0.5),
                transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
                transforms.ToTensor(),
                transforms.Normalize(mean, std)
            ])
        else:
            self.tf = transforms.Compose([
                transforms.Resize((img_size, img_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean, std)
            ])

    def __len__(self):
        return len(self.df)

    def _resolve_path(self, p):
        for prefix in [DATA_DIR, '/kaggle/input/csiro-biomass', '.']:
            cand = os.path.join(prefix, p)
            if os.path.exists(cand): return cand
            fname = os.path.basename(p)
            for sub in ['train', 'test']:
                cand2 = os.path.join(prefix, sub, fname)
                if os.path.exists(cand2): return cand2
        return p

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p = self._resolve_path(row['image_path'])
        bgr = cv2.imread(p)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        h, w = rgb.shape[:2]
        mid = w // 2
        l_np, r_np = rgb[:, :mid].copy(), rgb[:, mid:].copy()
        l_np = apply_white_balance_gray_world(l_np)
        r_np = apply_white_balance_gray_world(r_np)
        if self.is_train:
            l_np = apply_hsv_shadow_correction(l_np, 0.4)
            r_np = apply_hsv_shadow_correction(r_np, 0.4)
            if random.random() < 0.5: l_np, r_np = r_np, l_np
        t_l = self.tf(Image.fromarray(l_np))
        t_r = self.tf(Image.fromarray(r_np))
        item = {'img_l': t_l, 'img_r': t_r, 'sample_id': row.get('sample_id', str(idx))}
        if self.has_targets:
            item['targets'] = torch.tensor(self.targets[idx], dtype=torch.float32)
        return item


In [ ]:
# 5. DualStreamBiomassModel Architecture (DINOv2-Small)
class DualStreamBiomassModel(nn.Module):
    def __init__(self, backbone_name=CFG.BACKBONE, num_targets=3, fusion_dim=CFG.FUSION_DIM, dropout=CFG.DROPOUT, pretrained=True):
        super().__init__()
        kwargs = {'dynamic_img_size': True} if ('dinov2' in backbone_name or 'patch14' in backbone_name) else {}
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0, **kwargs)
        self.backbone_dim = self.backbone.num_features
        num_heads = 8 if self.backbone_dim % 8 == 0 else 4
        self.cross_view_attn = nn.MultiheadAttention(self.backbone_dim, num_heads, dropout=0.1, batch_first=True)
        self.attn_norm = nn.LayerNorm(self.backbone_dim)
        self.fusion_mlp = nn.Sequential(
            nn.Linear(self.backbone_dim * 2, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.reg_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fusion_dim, fusion_dim // 2),
                nn.LayerNorm(fusion_dim // 2),
                nn.GELU(),
                nn.Dropout(dropout * 0.5),
                nn.Linear(fusion_dim // 2, 64),
                nn.GELU(),
                nn.Linear(64, 1)
            ) for _ in range(num_targets)
        ])

    def extract_features(self, x):
        f = self.backbone(x)
        return f.mean(dim=1) if len(f.shape) == 3 else f.mean(dim=[2, 3]) if len(f.shape) == 4 else f

    def forward(self, img_l, img_r):
        f_l = self.extract_features(img_l)
        f_r = self.extract_features(img_r)
        tok = torch.stack([f_l, f_r], dim=1)
        attn_out, _ = self.cross_view_attn(tok, tok, tok)
        tok = self.attn_norm(tok + attn_out)
        fused = self.fusion_mlp(torch.cat([tok[:, 0], tok[:, 1]], dim=-1))
        return [F.softplus(h(fused)) for h in self.reg_heads]


In [ ]:
# 6. Fast 2-Stage Training & Stratified Cross-Validation
# 1. Load and pivot dataset
raw_df = pd.read_csv(TRAIN_CSV)
pivot_df = raw_df.pivot_table(
    index=['image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm'],
    columns='target_name', values='target', aggfunc='first'
).reset_index()
id_map = raw_df.drop_duplicates('image_path').set_index('image_path')['sample_id'].to_dict()
pivot_df['sample_id'] = pivot_df['image_path'].map(id_map)

skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
pivot_df['fold'] = -1
for f, (tr, va) in enumerate(skf.split(pivot_df, y=pivot_df['State'])):
    pivot_df.loc[va, 'fold'] = f

print(f'Training on {len(pivot_df)} clean samples across {CFG.N_FOLDS} folds.')

oof_preds = np.zeros((len(pivot_df), 5), dtype=np.float32)
oof_targets = pivot_df[CFG.ALL_TARGETS].values
fold_scores = []
criterion = nn.SmoothL1Loss()

for fold in range(CFG.N_FOLDS):
    print(f'\n========== FOLD {fold+1} / {CFG.N_FOLDS} ==========')
    tr_df = pivot_df[pivot_df['fold'] != fold].reset_index(drop=True)
    va_df = pivot_df[pivot_df['fold'] == fold].reset_index(drop=True)
    va_idx = pivot_df[pivot_df['fold'] == fold].index.values

    tr_loader = DataLoader(DualStreamDataset(tr_df, img_size=CFG.IMG_SIZE, is_train=True), batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    va_loader = DataLoader(DualStreamDataset(va_df, img_size=CFG.IMG_SIZE, is_train=False), batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    # Save sample batch for this fold
    import matplotlib.pyplot as plt
    sample_b = next(iter(tr_loader))
    imgs_l, imgs_r = sample_b['img_l'].cpu(), sample_b['img_r'].cpu()
    fig, axes = plt.subplots(min(len(imgs_l), 2), 2, figsize=(6, 6))
    mean = np.array([0.485, 0.456, 0.406]).reshape(1, 1, 3)
    std = np.array([0.229, 0.224, 0.225]).reshape(1, 1, 3)
    for i in range(min(len(imgs_l), 2)):
        rgb_l = np.clip(imgs_l[i].permute(1, 2, 0).numpy() * std + mean, 0, 1)
        rgb_r = np.clip(imgs_r[i].permute(1, 2, 0).numpy() * std + mean, 0, 1)
        axes[i, 0].imshow(rgb_l); axes[i, 0].axis('off'); axes[i, 0].set_title(f'Fold {fold+1} Left')
        axes[i, 1].imshow(rgb_r); axes[i, 1].axis('off'); axes[i, 1].set_title(f'Fold {fold+1} Right')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'sample_batch_fold{fold+1}.png'), bbox_inches='tight')
    plt.close()

    model = DualStreamBiomassModel(backbone_name=CFG.BACKBONE, num_targets=3, pretrained=True).to(CFG.DEVICE)
    scaler = torch.amp.GradScaler('cuda')
    best_r2 = -float('inf')
    best_preds = None

    # STAGE 1: Warm up heads (backbone frozen)
    for p in model.backbone.parameters(): p.requires_grad = False
    opt = AdamW([p for p in model.parameters() if p.requires_grad], lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    sched = CosineAnnealingLR(opt, T_max=CFG.STAGE1_EPOCHS, eta_min=1e-5)
    for ep in range(1, CFG.STAGE1_EPOCHS + 1):
        model.train()
        for b in tr_loader:
            opt.zero_grad()
            with torch.amp.autocast('cuda'):
                preds = model(b['img_l'].to(CFG.DEVICE), b['img_r'].to(CFG.DEVICE))
                loss = sum(criterion(p.squeeze(-1), b['targets'][:, i].to(CFG.DEVICE)) for i, p in enumerate(preds)) / 3.0
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
        sched.step()

    # STAGE 2: Full Fine-Tuning (backbone unfrozen)
    for p in model.backbone.parameters(): p.requires_grad = True
    opt = AdamW([
        {'params': model.backbone.parameters(), 'lr': CFG.LR * CFG.BACKBONE_LR_FACTOR},
        {'params': [p for n, p in model.named_parameters() if not n.startswith('backbone')], 'lr': CFG.LR}
    ], weight_decay=CFG.WEIGHT_DECAY)
    best_s2_r2 = -float('inf')
    s2_model_path = os.path.join(OUTPUT_DIR, f'best_s2_model_fold{fold+1}.pt')
    if CFG.STAGE2_WARMUP > 0 and CFG.STAGE2_EPOCHS > CFG.STAGE2_WARMUP:
        w_sched = LinearLR(opt, start_factor=0.1, total_iters=CFG.STAGE2_WARMUP)
        c_sched = CosineAnnealingLR(opt, T_max=CFG.STAGE2_EPOCHS - CFG.STAGE2_WARMUP, eta_min=CFG.LR * CFG.BACKBONE_LR_FACTOR * 0.05)
        sched = SequentialLR(opt, schedulers=[w_sched, c_sched], milestones=[CFG.STAGE2_WARMUP])
    else:
        sched = CosineAnnealingLR(opt, T_max=CFG.STAGE2_EPOCHS, eta_min=CFG.LR * CFG.BACKBONE_LR_FACTOR * 0.05)

    for ep in range(1, CFG.STAGE2_EPOCHS + 1):
        model.train()
        tr_loss = 0.0
        for b in tr_loader:
            opt.zero_grad()
            with torch.amp.autocast('cuda'):
                preds = model(b['img_l'].to(CFG.DEVICE), b['img_r'].to(CFG.DEVICE))
                loss = sum(criterion(p.squeeze(-1), b['targets'][:, i].to(CFG.DEVICE)) for i, p in enumerate(preds)) / 3.0
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            tr_loss += loss.item() * len(b['targets'])
        sched.step()

        # Validation with TTA
        model.eval()
        va_preds = []
        with torch.no_grad():
            for b in va_loader:
                l, r = b['img_l'].to(CFG.DEVICE), b['img_r'].to(CFG.DEVICE)
                p1 = model(l, r)
                p2 = model(torch.flip(r, [3]), torch.flip(l, [3]))
                p_avg = [(r1 + r2) * 0.5 for r1, r2 in zip(p1, p2)]
                va_preds.append(torch.cat(p_avg, dim=1).cpu().numpy())
        va_preds_3 = np.concatenate(va_preds, axis=0)
        va_preds_5 = derive_5_targets(va_preds_3)
        va_post = apply_2nd_place_postprocess(va_preds_5, states=va_df['State'].values)
        r2 = calculate_competition_r2(va_df[CFG.ALL_TARGETS].values, va_post)
        print(f'[Ep {ep:02d}] Train: {tr_loss/len(tr_df):.4f} | Val R2: {r2:.4f}')
        if r2 > best_s2_r2:
            best_s2_r2 = r2
            torch.save(model.state_dict(), s2_model_path)
        if r2 > best_r2:
            best_r2 = r2
            best_preds = va_post
            torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, f'best_model_fold{fold+1}.pt'))

    # STAGE 3: Head Calibration (backbone re-frozen)
    if CFG.STAGE3_EPOCHS > 0:
        print(f'--- [Fold {fold+1}] STAGE 3: Head Calibration ({CFG.STAGE3_EPOCHS} eps | Backbone RE-FROZEN) ---')
        load_p = s2_model_path if os.path.exists(s2_model_path) else os.path.join(OUTPUT_DIR, f'best_model_fold{fold+1}.pt')
        model.load_state_dict(torch.load(load_p, weights_only=True))
        for p in model.backbone.parameters(): p.requires_grad = False
        opt3 = AdamW([p for p in model.parameters() if p.requires_grad], lr=CFG.LR * 0.1, weight_decay=CFG.WEIGHT_DECAY)
        sched3 = CosineAnnealingLR(opt3, T_max=CFG.STAGE3_EPOCHS, eta_min=1e-6)
        for ep in range(1, CFG.STAGE3_EPOCHS + 1):
            model.train()
            tr_loss = 0.0
            for b in tr_loader:
                opt3.zero_grad()
                with torch.amp.autocast('cuda'):
                    preds = model(b['img_l'].to(CFG.DEVICE), b['img_r'].to(CFG.DEVICE))
                    loss = sum(criterion(p.squeeze(-1), b['targets'][:, i].to(CFG.DEVICE)) for i, p in enumerate(preds)) / 3.0
                scaler.scale(loss).backward()
                scaler.step(opt3); scaler.update()
                tr_loss += loss.item() * len(b['targets'])
            sched3.step()
            model.eval()
            va_preds = []
            with torch.no_grad():
                for b in va_loader:
                    l, r = b['img_l'].to(CFG.DEVICE), b['img_r'].to(CFG.DEVICE)
                    p1 = model(l, r)
                    p2 = model(torch.flip(r, [3]), torch.flip(l, [3]))
                    p_avg = [(r1 + r2) * 0.5 for r1, r2 in zip(p1, p2)]
                    va_preds.append(torch.cat(p_avg, dim=1).cpu().numpy())
            va_preds_3 = np.concatenate(va_preds, axis=0)
            va_preds_5 = derive_5_targets(va_preds_3)
            va_post = apply_2nd_place_postprocess(va_preds_5, states=va_df['State'].values)
            r2 = calculate_competition_r2(va_df[CFG.ALL_TARGETS].values, va_post)
            print(f'[S3 Ep {ep:02d}] Train: {tr_loss/len(tr_df):.4f} | Val R2: {r2:.4f}')
            if r2 > best_r2:
                best_r2 = r2
                best_preds = va_post
                torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, f'best_model_fold{fold+1}.pt'))
                print(f'  ★ New Best Model Saved for Fold {fold+1} (R2: {best_r2:.4f})')

    oof_preds[va_idx] = best_preds
    fold_scores.append(best_r2)
    print(f'Fold {fold+1} Best R2: {best_r2:.4f}')

total_oof_r2 = calculate_competition_r2(oof_targets, oof_preds)
print(f'\n========================================')
print(f'FINAL {CFG.N_FOLDS}-FOLD OOF COMPETITION R2: {total_oof_r2:.4f}')
print(f'Per-fold scores: {[round(s, 4) for s in fold_scores]}')
print(f'========================================')


In [ ]:
# 7. Dual-Stream Inference & Submission Generation
test_csv_path = CFG.TEST_CSV
if not os.path.exists(test_csv_path) and os.path.exists('/kaggle/input'):
    for root, _, files in os.walk('/kaggle/input'):
        if 'test.csv' in files:
            test_csv_path = os.path.join(root, 'test.csv')
            break

print(f'Reading test data from: {test_csv_path}')
test_df_raw = pd.read_csv(test_csv_path)
if 'target_name' in test_df_raw.columns:
    test_df_raw['clean_id'] = test_df_raw['sample_id'].astype(str).apply(lambda x: x.split('__')[0])
    unique_test = test_df_raw[['clean_id', 'image_path']].drop_duplicates().reset_index(drop=True)
else:
    unique_test = test_df_raw.copy()
    if 'clean_id' not in unique_test.columns: unique_test['clean_id'] = unique_test['sample_id']

test_ds = DualStreamBiomassDataset(
    unique_test, CFG.TEST_IMG_DIR, CFG.IMG_SIZE, is_training=False,
    camera_scaling_prob=0.0, strip_shuffle_prob=0.0, view_swap_prob=0.0, grayscale_prob=0.0
)
test_loader = DataLoader(test_ds, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=0)

model_checkpoints = sorted(glob.glob('best_model_fold*.pt'))
print(f'Found {len(model_checkpoints)} fold checkpoints for ensemble.')

all_fold_preds = []
for cp in model_checkpoints:
    model = DualStreamBiomassModel(CFG.BACKBONE, pretrained=False).to(DEVICE)
    model.load_state_dict(torch.load(cp, map_location=DEVICE))
    model.eval()
    
    f_preds = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f'Predicting {cp}', leave=False):
            img_l, img_r = batch['image_left'].to(DEVICE), batch['image_right'].to(DEVICE)
            # TTA: Standard + Horizontal flip
            r1, _ = model(img_l, img_r)
            r2, _ = model(torch.flip(img_r, [3]), torch.flip(img_l, [3]))
            avg_r = [(a + b) * 0.5 for a, b in zip(r1, r2)]
            f_preds.append(torch.cat(avg_r, dim=1).cpu().numpy())
    all_fold_preds.append(np.concatenate(f_preds, axis=0))

avg_raw = np.mean(all_fold_preds, axis=0)
avg_post = soft_physics_postprocess(avg_raw)

clean_ids = [s['sample_id'] for s in test_ds]
pred_dict = {
    clean_ids[i]: {col: avg_post[i, c_idx] for c_idx, col in enumerate(CFG.TARGET_ORDER)}
    for i in range(len(clean_ids))
}

submission_df = test_df_raw.copy()
submission_df['target'] = submission_df.apply(
    lambda r: pred_dict.get(r['clean_id'], {}).get(r['target_name'], 0.0), axis=1
)
sub = submission_df[['sample_id', 'target']]
sub.to_csv('submission.csv', index=False)
print('Submission saved to submission.csv. Preview:')
print(sub.head(10))
